# Introduction to Machine Learning
## A 90-Minute Journey for Actuaries

**Course Duration:** 90 minutes  
**Target Audience:** Actuaries and quantitative professionals  
**Learning Objective:** Understand core ML paradigms, model selection, and practical implementation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/YOUR_REPO/blob/main/02_intro_machine_learning.ipynb)

---

### What We'll Cover Today

This session introduces the fundamental concepts and practical tools for machine learning, with special emphasis on applications relevant to the actuarial profession. By the end, you will understand:

- The three learning paradigms and when to apply each
- How to select appropriate model complexity
- How to encode, scale, and engineer features
- How to build, evaluate, and optimize classification and regression models


In [ ]:
# Essential imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.metrics import precision_score, recall_score, f1_score
import warnings
warnings.filterwarnings('ignore')

# Import helper functions from mytools
from mytools import (
    generate_polynomial_data,
    generate_two_blobs,
    plot_model_selection_regression,
    plot_classification_boundaries,
    generate_multiclass_data,
    plot_confusion_matrix_custom
)

print("All imports successful!")


## 1. Learning Paradigms

Machine learning encompasses three primary paradigms, each suited to different types of problems:

### Three Core Paradigms

| **Paradigm** | **Description** | **Use Case** |
|---|---|---|
| **Supervised Learning** | Learn from labeled data: input X → output y | Predicting claims, customer churn |
| **Unsupervised Learning** | Find patterns in unlabeled data | Customer segmentation, anomaly detection |
| **Reinforcement Learning** | Learn through interaction & rewards | Optimal decision-making, control systems |

**In this session**, we focus on **Supervised Learning**, which splits into two main branches:

- **Classification:** Predict categorical outcomes (e.g., "Will claim occur?" → Yes/No)
- **Regression:** Predict continuous outcomes (e.g., "What will be the claim amount?" → €5,000)


## 2. Model Selection — Regression (Underfitting vs. Overfitting)

A critical aspect of machine learning is choosing the right model complexity. Too simple → underfitting. Too complex → overfitting. Let's visualize this with polynomial regression.

### The Bias-Variance Trade-off

- **Underfitting (High Bias):** Model is too simple to capture patterns → poor performance on both training and test data
- **Overfitting (High Variance):** Model memorizes noise → excellent training performance, poor test performance
- **Sweet Spot:** Balanced complexity that generalizes well to new data

Below, we generate 20 noisy samples from a cubic polynomial and fit models of varying complexity:


In [ ]:
# Generate sample data from a cubic polynomial with noise
X, y = generate_polynomial_data(n=20, noise=1.0, seed=42)

# Visualize the three regimes: underfitting, good fit, overfitting
plot_model_selection_regression(X, y)
plt.tight_layout()
plt.show()

print("Left: Linear model (underfitting) - captures the trend but misses curvature")
print("Middle: Cubic model (good fit) - reasonable complexity")
print("Right: Degree-19 polynomial (overfitting) - wiggles to fit every point")


### Key Insight: Polynomial Features as Feature Engineering

Notice how the cubic model captures the underlying pattern well. This is an example of **feature engineering**—we created polynomial features ($x^2$, $x^3$) from the original feature $x$. This is our first hint that domain knowledge and feature creation can substantially improve model performance.


## 3. Model Selection — Classification (Decision Boundaries)

Different algorithms learn different decision boundaries. Let's explore how four popular classifiers separate two blob classes:


In [ ]:
# Generate two-blob classification data
X, y = generate_two_blobs(n=300, seed=42)

# Define classifiers
classifiers = [
    DecisionTreeClassifier(random_state=42, max_depth=5),
    LogisticRegression(random_state=42, max_iter=1000),
    SVC(kernel='rbf', random_state=42),
    MLPClassifier(hidden_layer_sizes=(20, 20), max_iter=1000, random_state=42)
]

titles = [
    'Decision Tree',
    'Logistic Regression',
    'Support Vector Machine (RBF)',
    'Neural Network'
]

# Plot decision boundaries
plot_classification_boundaries(X, y, classifiers, titles, figsize=(16, 4))
plt.show()

print("Decision Tree: Axis-aligned splits, piecewise constant boundaries")
print("Logistic Regression: Linear boundary (can't capture non-linear patterns)")
print("SVM (RBF): Non-linear, smooth boundary")
print("Neural Network: Flexible, complex boundary")


### Observations

1. **Decision Tree** creates rectangular regions (axis-aligned splits)
2. **Logistic Regression** produces a linear boundary—works here, but not always
3. **Support Vector Machine** learns smooth, non-linear boundaries (RBF kernel)
4. **Neural Network** is highly flexible and can capture intricate patterns

The "best" algorithm depends on your data structure, interpretability needs, and computational constraints.


## 4. Data Encoding

Most machine learning algorithms require numerical inputs. Categorical variables must be encoded:

| **Feature Type** | **Encoding Method** | **Example** |
|---|---|---|
| **Nominal (unordered)** | One-Hot Encoding | Color: Red → [1,0,0], Blue → [0,1,0] |
| **Ordinal (ordered)** | Label/Map | Risk level: Low→1, Medium→2, High→3 |
| **Numerical** | No encoding | Age, Claim amount |

### One-Hot Encoding Example (Nominal)

```python
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

df = pd.DataFrame({'policy_type': ['car', 'home', 'car', 'life']})
df_encoded = pd.get_dummies(df, columns=['policy_type'])
```

### Ordinal Encoding Example

```python
df = pd.DataFrame({'risk_level': ['low', 'medium', 'high', 'low']})
risk_map = {'low': 1, 'medium': 2, 'high': 3}
df['risk_level_encoded'] = df['risk_level'].map(risk_map)
```


### Exercise 1: Data Encoding

**Task:** Encode a small DataFrame containing categorical features.


In [ ]:
# Sample insurance data
df_insurance = pd.DataFrame({
    'policy_type': ['car', 'home', 'car', 'life', 'home'],
    'risk_level': ['low', 'high', 'medium', 'low', 'high'],
    'age': [25, 45, 35, 60, 50]
})

print("Original DataFrame:")
print(df_insurance)
print()

# One-hot encode 'policy_type' using pd.get_dummies
df_encoded = pd.get_dummies(df_insurance, columns=['policy_type'], drop_first=False)

# Ordinal encode 'risk_level' using .map()
risk_mapping = {'low': 1, 'medium': 2, 'high': 3}
df_encoded['risk_level_encoded'] = df_encoded['risk_level'].map(risk_mapping)
df_encoded = df_encoded.drop('risk_level', axis=1)

print("Encoded DataFrame:")
print(df_encoded)


## 5. Scaling / Standardization

Many algorithms (e.g., SVM, Neural Networks, Logistic Regression) are sensitive to feature magnitude. Scaling brings features to comparable ranges.

### StandardScaler (Standardization)

$$z = \frac{x - \mu}{\sigma}$$

Transforms features to mean=0, standard deviation=1.

### MinMaxScaler (Normalization)

$$x' = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$$

Scales features to the range [0, 1].

### Code Example


In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Sample data
X_sample = np.array([[10, 1000],
                      [20, 2000],
                      [30, 3000]])

print("Original data:")
print(X_sample)
print()

# Standardization
scaler_standard = StandardScaler()
X_standardized = scaler_standard.fit_transform(X_sample)
print("After StandardScaler (mean≈0, std≈1):")
print(X_standardized)
print()

# Normalization
scaler_minmax = MinMaxScaler()
X_normalized = scaler_minmax.fit_transform(X_sample)
print("After MinMaxScaler (range [0,1]):")
print(X_normalized)


### Exercise 2: Apply Scaling to Sample Data

**Task:** Scale a dataset using both StandardScaler and MinMaxScaler, then compare.


In [ ]:
# Create sample premium data
X_sample = np.array([[500, 25],
                      [1500, 45],
                      [2000, 60],
                      [750, 35]])

print("Original premium and age data:")
print(X_sample)
print()

# Create a StandardScaler, fit and transform X_sample
scaler_std = StandardScaler()
X_std = scaler_std.fit_transform(X_sample)
print("After StandardScaler:")
print(X_std)
print()

# Create a MinMaxScaler, fit and transform X_sample
scaler_minmax = MinMaxScaler()
X_minmax = scaler_minmax.fit_transform(X_sample)
print("After MinMaxScaler:")
print(X_minmax)

print("Note: Scaling preserves relationships but changes magnitudes.")


## 6. Feature Engineering

**Feature engineering** is the art of creating new features from raw data. It often has bigger impact on model performance than algorithm choice.

### We've Already Seen Feature Engineering!

In Section 2, we created polynomial features ($x$, $x^2$, $x^3$) from a single input. This transformation allowed a cubic model to fit the data.

### Actuarial Examples

- **Combined Features:** `claim_ratio = claim_amount / vehicle_value`
- **Polynomial Features:** `age^2` to capture non-linear age effects
- **Interactions:** `premium_group * vehicle_type` to capture combined effects
- **Domain Ratios:** `average_claim_per_policy = total_claims / policy_count`

### Key Principle

**Domain knowledge is your best tool.** Understand your data, consult with subject matter experts, and craft features that capture meaningful patterns.


## 7. Train/Test Split

A fundamental principle: evaluate on data the model has **never seen** before.

### The Workflow

1. Split data: typically 80% training, 20% testing
2. Fit the model on training data
3. Evaluate on test data

### Critical: Avoid Data Leakage

**Data Leakage** occurs when test set information "leaks" into training:

```python
# ❌ WRONG: Scaling before split
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)  # Fit on ALL data
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2)
```

```python
# ✓ CORRECT: Scaling after split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)      # Fit only on training
X_test_scaled = scaler.transform(X_test)            # Transform test using training statistics
```


In [ ]:
# Example: correct train/test split and scaling
from sklearn.datasets import load_iris

iris = load_iris()
X_iris = iris.data
y_iris = iris.target

# Step 1: Split first
X_train, X_test, y_train, y_test = train_test_split(
    X_iris, y_iris, test_size=0.2, random_state=42
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")
print()

# Step 2: Scale using only training statistics
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✓ Data correctly split and scaled (no leakage)")


## 8. Evaluation Metrics

The choice of metric depends on your problem and business objectives.

### Regression: Root Mean Squared Error (RMSE)

$$\text{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}$$

### Classification Metrics

For a binary classification problem (e.g., claim/no-claim):

- **Accuracy:** $\frac{\text{TP} + \text{TN}}{\text{Total}}$ — Overall correctness
- **Precision:** $\frac{\text{TP}}{\text{TP} + \text{FP}}$ — Of predicted positives, how many are correct?
- **Recall:** $\frac{\text{TP}}{\text{TP} + \text{FN}}$ — Of actual positives, how many did we find?
- **F1-Score:** $2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$ — Harmonic mean

### When to Use Each

| **Metric** | **When to Use** |
|---|---|
| Accuracy | Balanced classes, general purpose |
| Precision | Minimize false positives (e.g., insurance fraud detection) |
| Recall | Minimize false negatives (e.g., disease diagnosis) |
| F1-Score | Need a balance between precision and recall |

### Code Example


In [ ]:
from sklearn.metrics import classification_report, accuracy_score

# Example predictions vs. actual labels
y_true = np.array([0, 0, 1, 1, 0, 1, 1, 0, 1, 1])
y_pred = np.array([0, 0, 1, 0, 0, 1, 1, 1, 1, 1])

print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=['No Claim', 'Claim']))
print()

# Individual metrics
acc = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print(f"Accuracy: {acc:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1-Score: {f1:.3f}")


## 9. Decision Trees with scikit-learn

Decision Trees are interpretable, easy to use, and effective across many domains.

### The sklearn Interface

All sklearn estimators follow a consistent interface:

```python
clf = SomeClassifier()          # Create instance
clf.fit(X_train, y_train)        # Fit to training data
y_pred = clf.predict(X_test)     # Make predictions
score = clf.score(X_test, y_test) # Evaluate
```

### Exercise 3: Train and Evaluate a Decision Tree

**Task:** Train a Decision Tree classifier on the blob data, split it properly, and generate a classification report.


In [ ]:
# Use the two-blob data from earlier
X, y = generate_two_blobs(n=300, seed=42)

# Step 1: Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 2: Create and train a Decision Tree
tree_clf = DecisionTreeClassifier(max_depth=5, random_state=42)
tree_clf.fit(X_train, y_train)

# Step 3: Make predictions
y_pred = tree_clf.predict(X_test)

# Step 4: Evaluate
print("Decision Tree - Classification Report:")
print(classification_report(y_test, y_pred))
print()
print(f"Test Accuracy: {tree_clf.score(X_test, y_test):.3f}")


## 10. Random Forests

A **Random Forest** is an ensemble of many decision trees. Instead of relying on a single tree, we:

1. Build multiple trees on bootstrap samples of the data
2. Introduce randomness (e.g., random feature subsets for splits)
3. Average predictions across all trees

### Key Advantages

- **Reduced Overfitting:** Averaging multiple trees is more robust than a single tree
- **Black-box Model:** Often performs very well, but less interpretable
- **Parallelizable:** Trees are independent and can be trained in parallel

### Comparison: Single Tree vs. Random Forest


In [ ]:
# Use the blob data
X, y = generate_two_blobs(n=300, seed=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Single Decision Tree
tree_clf = DecisionTreeClassifier(max_depth=5, random_state=42)
tree_clf.fit(X_train, y_train)
tree_acc = tree_clf.score(X_test, y_test)

# Random Forest
rf_clf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_clf.fit(X_train, y_train)
rf_acc = rf_clf.score(X_test, y_test)

print(f"Decision Tree Test Accuracy: {tree_acc:.3f}")
print(f"Random Forest Test Accuracy: {rf_acc:.3f}")
print()
print("Random Forest Benefits:")
print("- Often achieves better accuracy")
print("- More robust to overfitting")
print("- Can handle large feature sets")
print("- Slower to train than single tree, but still fast for most datasets")


## 11. Hyperparameter Optimization with GridSearchCV

**Hyperparameters** are settings you choose before training (e.g., `max_depth`, `n_estimators`). Finding good values is crucial.

**GridSearchCV** systematically tries combinations and uses cross-validation to estimate performance.

### Example: Optimizing Random Forest


In [ ]:
from sklearn.model_selection import GridSearchCV

# Use the blob data
X, y = generate_two_blobs(n=300, seed=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define a grid of hyperparameters to search
param_grid = {
    'max_depth': [3, 5, 7, 10],
    'n_estimators': [50, 100, 200]
}

# Create base classifier
rf = RandomForestClassifier(random_state=42)

# GridSearchCV tries all combinations
grid_search = GridSearchCV(rf, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best cross-validation score: {grid_search.best_score_:.3f}")
print(f"Test set score with best params: {grid_search.score(X_test, y_test):.3f}")


### What's Happening?

1. GridSearchCV creates a 4 × 3 = 12 combinations of hyperparameters
2. For each, it uses 5-fold cross-validation on the training set
3. It selects the combination with the best average CV score
4. The best estimator is retrained on all training data and returned
5. We evaluate the final model on the held-out test set

**Tip:** Use GridSearchCV inside your train/test pipeline to avoid data leakage.


## 12. Exercise: Comprehensive Classification Pipeline

**Task:** Build classification reports for multiple algorithms on the blob dataset.

You will:
1. Generate two-blob data
2. Split into train/test
3. Train Logistic Regression, SVM, and Neural Network
4. Evaluate each with classification_report
5. Compare performance

Below is a template with some parts pre-filled. Complete the missing parts (marked `# TODO`).


In [ ]:
# Step 1: Generate and split data
X, y = generate_two_blobs(n=300, seed=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 2: Scale the data (important for LR, SVM, NN)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Step 3: Define classifiers
classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Support Vector Machine': SVC(kernel='rbf', random_state=42),
    'Neural Network': MLPClassifier(hidden_layer_sizes=(20, 20), max_iter=1000, random_state=42)
}

# Step 4: Train and evaluate each
results = {}
for name, clf in classifiers.items():
    # TODO: Fit the classifier
    clf.fit(X_train_scaled, y_train)
    
    # TODO: Make predictions
    y_pred = clf.predict(X_test_scaled)
    
    # TODO: Store accuracy
    acc = accuracy_score(y_test, y_pred)
    results[name] = acc
    
    # Print classification report
    print(f"\n{'='*50}")
    print(f"{name}")
    print(f"{'='*50}")
    print(f"Test Accuracy: {acc:.3f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=['Class 0', 'Class 1']))

print(f"\n{'='*50}")
print("Summary of Accuracies:")
print(f"{'='*50}")
for name, acc in results.items():
    print(f"{name:30s}: {acc:.3f}")


### Reflection

- Which classifier performed best? Why might that be?
- Did scaling help? (Try commenting out the scaling step for SVM and NN—what happens?)
- What trade-offs exist between accuracy and interpretability?
- In an actuarial context, which metric (accuracy, precision, recall) might matter most?


---

## Summary

In this 90-minute session, you have learned:

1. **Learning Paradigms:** Supervised (classification & regression), Unsupervised, Reinforcement
2. **Model Complexity:** Balance between underfitting (bias) and overfitting (variance)
3. **Data Preparation:** Encoding, scaling, feature engineering
4. **Workflow Best Practices:** Train/test split, avoiding data leakage
5. **Evaluation:** Metrics for regression and classification
6. **Algorithms:** Decision Trees, Random Forests, Logistic Regression, SVM, Neural Networks
7. **Optimization:** Using GridSearchCV to tune hyperparameters

### Key Takeaways for Actuaries

- Machine learning can enhance reserve prediction, pricing, and risk assessment
- Careful data preparation (encoding, scaling, feature engineering) often matters more than algorithm choice
- Always evaluate on held-out test data
- Interpretability is often valued in actuarial applications—consider decision trees and logistic regression
- Domain expertise drives feature engineering—collaborate with your risk team

### Next Steps

- Explore real actuarial datasets
- Experiment with different feature engineering strategies
- Consider your specific business objectives when choosing evaluation metrics
- Stay curious and iterate!

---

**Created for:** Introduction to Machine Learning for Actuaries  
**Duration:** 90 minutes  
**Tools:** Python, scikit-learn, pandas, matplotlib
